### Chapter 7 — Using MCP with Real Production Agents (LangGraph + MCP)

Since you already understand agents and ReAct, and you specifically want **LangGraph**, this chapter will show how MCP integrates with **LangGraph-based agents** in production.

LangGraph is ideal because MCP tools are **dynamic**, and LangGraph is built for **stateful, controllable agent workflows**.

---

#### 1️⃣ Why LangGraph + MCP is the Best Combination

<img src="https://miro.medium.com/v2/resize%3Afit%3A1200/0%2Afmk9z4CULIH1fd9w.png" width="800"/>

<img src="https://miro.medium.com/v2/resize%3Afit%3A1400/1%2AffzadZCKXJT7n4JaRVFvcQ.jpeg" width="800"/>

<img src="https://miro.medium.com/v2/resize%3Afit%3A1200/0%2AIgOnp2b7ze0YFCxj.png" width="800"/>

<img src="https://blog.langchain.com/content/images/2024/01/simple_multi_agent_diagram--1-.png" width="800"/>

LangGraph gives you:

* Stateful execution
* Deterministic control
* Graph-based workflows
* Multi-agent orchestration

MCP gives you:

* Dynamic tools
* Shared tool ecosystem
* Tool microservices
* Scalable tool infrastructure

Together:

```text
LangGraph = Agent brain orchestration
MCP = Tool infrastructure layer
```

---

# 2️⃣ High-Level Architecture (LangGraph + MCP)

Architecture becomes:

```text
User
 ↓
LangGraph Graph
 ↓
LLM Node
 ↓
MCP Client Node
 ↓
MCP Server
 ↓
Tool Execution
```

LangGraph controls flow.

MCP executes tools.

---

# 3️⃣ State Design in LangGraph (Important)

LangGraph works using shared state.

Example:

```python
from typing import TypedDict, List, Any

class AgentState(TypedDict):
    messages: List[dict]
    tool_calls: List[dict]
    tool_results: List[Any]
```

State flows between nodes.

---

# 4️⃣ Creating MCP Client Node in LangGraph

This node executes MCP tools.

```python
import asyncio
from mcp.client.session import ClientSession
from mcp.client.stdio import stdio_client

class MCPToolExecutor:

    def __init__(self):
        self.session = None

    async def connect(self):
        self.stdio = await stdio_client("python server.py")
        read, write = self.stdio
        self.session = await ClientSession(read, write).__aenter__()

    async def execute(self, tool_name, args):

        result = await self.session.call_tool(
            tool_name,
            args
        )

        return result
```

This becomes LangGraph node.

---

# 5️⃣ LLM Node (Tool Selection)

This node decides which MCP tool to call.

```python
def llm_node(state: AgentState):

    messages = state["messages"]

    response = llm.invoke(messages)

    return {
        "messages": messages + [response],
        "tool_calls": response.tool_calls
    }
```

Same ReAct pattern.

But tools come from MCP.

---

# 6️⃣ MCP Execution Node

Executes tool using MCP client.

```python
async def mcp_node(state: AgentState):

    tool_calls = state["tool_calls"]

    results = []

    for call in tool_calls:

        result = await mcp_executor.execute(
            call["name"],
            call["args"]
        )

        results.append(result)

    return {
        "tool_results": results
    }
```

This integrates MCP.

---

# 7️⃣ Building LangGraph Workflow

```python
from langgraph.graph import StateGraph, END

builder = StateGraph(AgentState)

builder.add_node("llm", llm_node)
builder.add_node("tool_executor", mcp_node)

builder.add_edge("llm", "tool_executor")
builder.add_edge("tool_executor", "llm")

builder.set_entry_point("llm")

graph = builder.compile()
```

This creates ReAct loop using MCP tools.

---

# 8️⃣ Full Flow (Step-by-Step)

Execution:

```text
Step 1: User input enters LangGraph

Step 2: LLM node decides tool

Step 3: MCP node executes tool

Step 4: Tool result added to state

Step 5: LLM continues reasoning

Step 6: Repeat until finished
```

Same ReAct loop.

But tool execution handled by MCP.

---

# 9️⃣ Dynamic Tool Discovery in LangGraph

At startup:

```python
tools = await session.list_tools()
```

Convert to LLM tool schema:

```python
llm_tools = [
    {
        "name": tool.name,
        "description": tool.description,
        "parameters": tool.inputSchema
    }
]
```

Pass to LLM.

Now LLM knows MCP tools dynamically.

---

# 10️⃣ Production-Level Architecture

Real system:

```text
LangGraph Agent
   ↓
MCP Client Layer
   ↓
Multiple MCP Servers:

   filesystem server
   database server
   github server
   internal tools server
```

Each server independent.

LangGraph orchestrates reasoning.

MCP executes tools.

---

# 11️⃣ Multi-Agent Architecture with LangGraph + MCP

![Image](https://miro.medium.com/0%2ACP98j5HdhmC-nTON.jpg)

![Image](https://learn.microsoft.com/en-us/azure/architecture/ai-ml/guide/_images/magentic-pattern-example.svg)

![Image](https://microservices.io/i/Microservice_Architecture.png)

![Image](https://miro.medium.com/1%2AJIDAhbsGGTztmcJ6OxNkrg.png)

Example:

```text
Supervisor Agent (LangGraph)
   ├── Coding Agent
   ├── Data Agent
   ├── Email Agent

All connected to same MCP servers
```

Shared tool ecosystem.

No duplication.

---

# 12️⃣ Real Production Example (Coding Agent)

Coding agent using LangGraph + MCP:

LangGraph handles:

* reasoning
* planning
* execution flow

MCP handles:

* read file
* write file
* git commit
* run tests

Clean separation.

---

# 13️⃣ Best Practices (Very Important)

## 1. Keep MCP client separate

Good:

```text
LangGraph → MCP Client → MCP Server
```

Bad:

```text
LangGraph directly executing tools
```

---

## 2. Connect MCP once at startup

Not every tool call.

Reuse session.

---

## 3. Cache tool schemas

Avoid repeated discovery.

---

## 4. Use multiple MCP servers

Split by domain:

```text
filesystem MCP server
database MCP server
external API MCP server
```

---

# 14️⃣ Full Production Mental Model

Final architecture:

```text
LangGraph → reasoning orchestration
MCP Client → tool communication layer
MCP Servers → tool providers
Tools → execution layer
```

This is modern AI architecture.

Used in:

* IDE agents
* Autonomous agents
* Enterprise AI systems

---

# Final Key Insight

LangGraph controls:

Flow, reasoning, orchestration

MCP controls:

Tool discovery, execution, infrastructure

Together they create:

Fully scalable agent systems.

---

# Final Analogy — Company Structure 🏢

Think of company:

CEO → LangGraph agent
Managers → Nodes
Employees → MCP tools
Departments → MCP servers

CEO doesn't do work directly.

CEO delegates to departments.

Departments execute work.

Same architecture.

---

# You Have Now Learned MCP Completely 🎉

You now understand:

* Why MCP exists
* Architecture
* Tools, resources, prompts
* MCP server implementation
* MCP client implementation
* LangGraph integration
* Production architecture

